In [2]:
!pip install -q ultralytics

In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

In [6]:
!hf download HuggingFaceFV/finevideo --repo-type dataset --include "data/train-00001-of-01357.parquet" --local-dir finevideo

Hint: A new version of huggingface_hub (2.4.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
Reconstructing (incomplete total...): |            |  0.00B /  0.00B            

Fetching 1 files:   0%|                                   | 0/1 [00:00<?, ?it/s]
Reconstructing (incomplete total...): 100%|████████|  566MB /  566MB,  134MB/s  

Fetching 1 files: 100%|███████████████████████████| 1/1 [00:03<00:00,  3.03s/it]
Download complete: █████████████████████████████████████████|  565MB,  485MB/s  
Reconstruction complete: 100%|█████████████████████|  566MB /  566MB,  205MB/s  ✓ Downloaded
  path: /kaggle/working/finevideo
Download complete: █████████████████████████████████████████|  565MB,  485MB/s  
Reconstruction complete: 100%|█████████████████████|  566MB /  566MB,  205MB/s  


In [9]:
import pandas as pd
df = pd.read_parquet("finevideo/data/train-00001-of-01357.parquet")
print(df.shape)

(33, 2)


In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11l.pt")

In [13]:
# so we took 3 videos, extracted their frames using opencv and passed them into YOLO, then yolo returned names of object it recognized
import cv2

for video_idx in [0, 1, 2]:
    video_bytes = df.iloc[video_idx]['mp4']
    with open("sample.mp4", "wb") as f:
        f.write(video_bytes)

    cap = cv2.VideoCapture("sample.mp4")
    frame_num = 0
    found_any = False

    while frame_num < 60:
        ret, frame = cap.read()
        if not ret:
            break

        timestamp_sec = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0

        result = model(frame, conf=0.4, verbose=False)[0]
        names_found = [model.names[int(c)] for c in result.boxes.cls]
        if names_found:
            print(f"video {video_idx}, frame {frame_num}, t={timestamp_sec:.3f}s: {names_found}")
            found_any = True

        frame_num += 1

    cap.release()
    if not found_any:
        print(f"video {video_idx}: nothing found in first 60 frames")

video 0, frame 0, t=0.000s: ['person']
video 0, frame 1, t=0.033s: ['person']
video 0, frame 2, t=0.067s: ['person']
video 0, frame 3, t=0.100s: ['person']
video 0, frame 4, t=0.133s: ['person']
video 0, frame 5, t=0.167s: ['person']
video 0, frame 6, t=0.200s: ['person']
video 0, frame 8, t=0.267s: ['person']
video 0, frame 9, t=0.300s: ['person']
video 0, frame 10, t=0.333s: ['person']
video 0, frame 11, t=0.367s: ['person']
video 0, frame 12, t=0.400s: ['person']
video 0, frame 13, t=0.433s: ['person']
video 0, frame 14, t=0.467s: ['person']
video 0, frame 15, t=0.500s: ['person']
video 0, frame 18, t=0.600s: ['person']
video 0, frame 19, t=0.633s: ['person']
video 0, frame 20, t=0.667s: ['person']
video 0, frame 21, t=0.700s: ['person']
video 0, frame 22, t=0.733s: ['person']
video 0, frame 23, t=0.767s: ['person']
video 0, frame 24, t=0.800s: ['person']
video 0, frame 25, t=0.833s: ['person']
video 0, frame 26, t=0.867s: ['person']
video 0, frame 27, t=0.900s: ['person']
video 0, 

In [14]:
# since its returning other objects also apart from our given target, we gonna filter them 

target_classes = {"person", "car", "dog", "laptop", "cell phone"}

# Keeping only 5 targets one
names_found = [
    name for name in names_found
    if name in target_classes
]
#video_index — which video we're processing.
# frame_index — which frame we're processing.
# names_found — objects detected in that frame.
print(f"video {video_idx}, frame {frame_num}: {names_found}")


video 2, frame 60: []


First, we need to figure out how to get timestamps from the given data. Can we use simple maths with the frame num and FPS if the video has a constant frame rate? and if its variable frame rate (VFR), then what do we do?

For this, We simply used the OPENCV function CAP_PROP_POS_MSEC to get the timestamp of each frame after reading it, and converted it from ms to seconds. Now we need to check if these timestamps are reliable, especially for VFR videos, before we use them for bucketing.
                                                                                                                                                                                                                                                            
                                                                                                                                                                                                                                                            

In [15]:
# testing if the timestamps seems logical and not jumping around or going backward
import cv2

for video_idx in [0, 1, 2]:
    video_bytes = df.iloc[video_idx]['mp4']
    with open("sample.mp4", "wb") as f:
        f.write(video_bytes)

    cap = cv2.VideoCapture("sample.mp4")
    timestamps = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        timestamps.append(cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0)

    cap.release()

    deltas = [timestamps[i+1] - timestamps[i] for i in range(len(timestamps) - 1)]
    print(f"video {video_idx}: {len(timestamps)} frames")
    print(f"  monotonic increasing: {all(d > 0 for d in deltas)}")
    print(f"  min delta: {min(deltas):.4f}s, max delta: {max(deltas):.4f}s")

video 0: 929 frames
  monotonic increasing: True
  min delta: 0.0333s, max delta: 0.0333s
video 1: 14827 frames
  monotonic increasing: True
  min delta: 0.0333s, max delta: 0.0333s
video 2: 18166 frames
  monotonic increasing: True
  min delta: 0.0334s, max delta: 0.0334s


In [16]:
# testing more videos if the timestamps seems logical and not jumping around or going backward
import cv2

results = []

for video_idx in range(len(df)):          # all 33 videos in this file
    video_bytes = df.iloc[video_idx]['mp4']
    with open("sample.mp4", "wb") as f:
        f.write(video_bytes)

    cap = cv2.VideoCapture("sample.mp4")
    timestamps = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        timestamps.append(cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0)
    cap.release()

    deltas = [timestamps[i+1] - timestamps[i] for i in range(len(timestamps) - 1)]
    is_monotonic = all(d > 0 for d in deltas)
    min_d, max_d = min(deltas), max(deltas)

    results.append((video_idx, len(timestamps), is_monotonic, min_d, max_d))
    print(f"video {video_idx}: {len(timestamps)} frames, monotonic={is_monotonic}, delta range=[{min_d:.4f}, {max_d:.4f}]")

# summary: anything that failed
bad = [r for r in results if not r[2] or (r[4] - r[3]) > 0.01]
print(f"\n{len(bad)} videos out of {len(results)} look suspicious" if bad else "\nAll videos clean — monotonic with tight, consistent deltas")

video 0: 929 frames, monotonic=True, delta range=[0.0333, 0.0333]
video 1: 14827 frames, monotonic=True, delta range=[0.0333, 0.0333]
video 2: 18166 frames, monotonic=True, delta range=[0.0334, 0.0334]
video 3: 3654 frames, monotonic=True, delta range=[0.0417, 0.0417]
video 4: 10817 frames, monotonic=True, delta range=[0.0417, 0.0417]
video 5: 1181 frames, monotonic=True, delta range=[0.0333, 0.0333]
video 6: 17376 frames, monotonic=True, delta range=[0.0333, 0.0333]
video 7: 7805 frames, monotonic=True, delta range=[0.0417, 0.0417]
video 8: 8595 frames, monotonic=True, delta range=[0.0400, 0.0400]
video 9: 12074 frames, monotonic=True, delta range=[0.0336, 0.0336]
video 10: 4785 frames, monotonic=True, delta range=[0.0333, 0.0333]
video 11: 1771 frames, monotonic=True, delta range=[0.0334, 0.0334]
video 12: 957 frames, monotonic=True, delta range=[0.0417, 0.0417]
video 13: 8835 frames, monotonic=True, delta range=[0.0417, 0.0417]
video 14: 9498 frames, monotonic=True, delta range=[0.0

now we gonna create one bucket for every second, for each decoded frame, we will use timestamp to determine which buckets it belongs to , like 0.0-0.999s timestamp wil be in bucket 0 and so on

for each frame, we will store the detected target objects without any dups and count of people detected in that frame. once all frames are grouped into their respective buckets, we will apply the assignment rules: include an object if it appears in at least half of the frames in that bucket, and calculate the median people count, rounding down when needed.

In [17]:

import cv2
from collections import defaultdict

target_classes = {"person", "car", "dog", "laptop", "cell phone"}

for video_idx in [0, 1, 2]:

    video_bytes = df.iloc[video_idx]["mp4"]
    with open("sample.mp4", "wb") as f:
        f.write(video_bytes)

    cap = cv2.VideoCapture("sample.mp4")

    # each second will have its own bucket containing frame info
    # bucket index represents the second. each entry stores the objects detected in a frame and the number of people detected
    buckets = defaultdict(list)

    frame_num = 0

    #capping it to 60 just for testing purpose before we do full run
    while frame_num < 60:
        ret, frame = cap.read()

        if not ret:
            break

        # getting the timestamp of the current frame in seconds
        timestamp_sec = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0

        # passing the frame to YOLO to detect objects
        result = model(frame, conf=0.4, verbose=False)[0]

        # getting the names of all objects detected in this frame
        names_found = [
            model.names[int(c)]
            for c in result.boxes.cls
        ]

        # keeping only the 5 objects required in our assignment
        names_found = [
            name for name in names_found
            if name in target_classes
        ]

        # finding which second this frame belongs to
        bucket_idx = int(timestamp_sec)

        # counting how many people YOLO detected in this frame
        person_count_this_frame = sum(
            1 for c in result.boxes.cls
            if model.names[int(c)] == "person"
        )

        # removing duplicate object names because we only need to know
        # whether an object appeared in this frame or not
        objects_this_frame = set(names_found)

        # storing this frame's objects and people count in its bucket
        buckets[bucket_idx].append(
            (objects_this_frame, person_count_this_frame)
        )

        frame_num += 1

    cap.release()

    # checking what we have stored in each bucket
    print(f"\nVideo {video_idx}")

    for bucket_idx, records in sorted(buckets.items()):
        print(f"Bucket {bucket_idx}: {records}")



Video 0
Bucket 0: [({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), (set(), 0), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), (set(), 0), (set(), 0), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), (set(), 0), ({'person'}, 1)]
Bucket 1: [({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), (set(), 0), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), ({'person'}, 1), (set(), 0), (set(), 0), (set(), 0), (set(), 0), (set(), 0), (set(), 0), (set(), 0), (set(), 0), (set(), 0)]

Video 1
Bucket 0: 

Now, right now, we are storing information about every frame separately.But we wants ONE final result per second.
so for it we aggregates it
for objects, we check how many frames an object appeared in and only keep it if it appeared in atleast 50% of the frames. for people, we take the person count from each frame, find the median and round it down if needed

In [ ]:
def aggregate_bucket(records):
    # records = list of (objects_set, person_count) tuples for one bucket
    total_frames = len(records)

    # objects: keep only those that appeared in at least 50% of frames
    all_classes_seen = set().union(*(objs for objs, _ in records)) if records else set()
    objects_present = [
        cls for cls in all_classes_seen
        if sum(1 for objs, _ in records if cls in objs) / total_frames >= 0.5
    ]

    # people: median of per-frame counts, rounded down if it falls between two numbers
    person_counts = sorted(p for _, p in records)
    n = len(person_counts)
    mid = n // 2
    if n % 2 == 1:
        median_people = person_counts[mid]
    else:
        median_people = (person_counts[mid - 1] + person_counts[mid]) // 2  # // floors automatically

    return sorted(objects_present), median_people

In [ ]:
final_results = []
# final loop that integrates above func
for bucket_idx in sorted(buckets.keys()):
    records = buckets[bucket_idx]
    objects, people = aggregate_bucket(records)
    final_results.append({
        "second": bucket_idx,
        "objects": objects,
        "people": people
    })

for r in final_results:
    print(r)

now we need to save our aggregated results in the required JSONL format, where each line represents one second of a video. the video field will contain the parquet filename and row number (like filename:0) instead of just the video index. we'll use python's json module to convert our dictionaries into JSON lines and append them to results.jsonl. also, we'll add error handling so if one video fails, the rest can still be processed.

In [20]:
import cv2
import json
from collections import defaultdict

video_filename = "train-00001-of-01357.parquet"

for video_idx in [0, 1, 2]:

    buckets = defaultdict(list)
    cap = None

    try:
        video_bytes = df.iloc[video_idx]["mp4"]

        with open("sample.mp4", "wb") as f:
            f.write(video_bytes)

        cap = cv2.VideoCapture("sample.mp4")

        if not cap.isOpened():
            raise ValueError("Could not open video")

        frame_num = 0

        while frame_num < 60:
            ret, frame = cap.read()

            if not ret:
                break

            # getting the timestamp of the current frame in seconds
            timestamp_sec = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0

            # passing the frame to YOLO to detect objects
            result = model(frame, conf=0.4, verbose=False)[0]

            # getting the names of all objects detected in this frame
            names_found = [
                model.names[int(c)]
                for c in result.boxes.cls
            ]

            # keeping only the 5 objects required in our assignment
            names_found = [
                name for name in names_found
                if name in target_classes
            ]

            # finding which second this frame belongs to
            bucket_idx = int(timestamp_sec)

            # counting how many people YOLO detected in this frame
            person_count_this_frame = sum(
                1 for c in result.boxes.cls
                if model.names[int(c)] == "person"
            )

            # removing duplicate object names because we only need to know
            # whether an object appeared in this frame or not
            objects_this_frame = set(names_found)

            # storing this frame's objects and people count in its bucket
            buckets[bucket_idx].append(
                (objects_this_frame, person_count_this_frame)
            )

            frame_num += 1

    except Exception as e:

        # if something goes wrong, saving the error so we can still process the next video
        error_message = str(e)
        print(f"Video {video_idx} failed: {error_message}")

    finally:

        # closing the video after processing it
        if cap is not None:
            cap.release()

        # now combining all frame info in each bucket into one result per second
        if buckets:
            final_results = []

            for bucket_idx, records in sorted(buckets.items()):

                # getting the final objects and median people count for this second
                objects_present, median_people = aggregate_bucket(records)

                # creating one final entry for this second
                record = {
                    "video": f"{video_filename}:{video_idx}",
                    "second": bucket_idx,
                    "objects": objects_present,
                    "people": median_people
                }

                final_results.append(record)

            # writing each second's result into results.jsonl
            with open("results.jsonl", "a") as out_file:
                for record in final_results:
                    out_file.write(json.dumps(record) + "\n")

        # writing an error record if the video failed
        if 'error_message' in locals():
            with open("results.jsonl", "a") as out_file:
                error_record = {
                    "video": f"{video_filename}:{video_idx}",
                    "error": error_message
                }
                out_file.write(json.dumps(error_record) + "\n")

            del error_message

        else:
            print(f"Video {video_idx} processed successfully!")

Video 0 processed successfully!
Video 1 processed successfully!
Video 2 processed successfully!


Next step is batching or parallelizing the whole run
so for it, i'm gonna apply my distributed and parallel computing knowledge. As far as i understand, we can only parallelize stuff that dont depend on each other.
so we have two options in here, do parallize video wise or frame wise within one video 

frames within one video are independent since YOLO treats each frame as a separate image, so we can batch multiple frames together and process them in one GPU call. videos are also independent, but since we only have one GPU, running multiple video processes at once wont really give us a speedup and can cause extra overhead and memory issues. instead, we can process videos one by one and batch frames within each video. we can also overlap CPU decoding with GPU inference, but for now batching is the main optimization we should focus on.

In [21]:
import cv2
import json
from collections import defaultdict

video_filename = "train-00001-of-01357.parquet"
batch_size = 16

for video_idx in [0, 1, 2]:

    buckets = defaultdict(list)
    cap = None

    def process_batch(frame_batch, timestamp_batch):

        # passing all frames in this batch to YOLO in one call
        results = model(frame_batch, conf=0.4, verbose=False)

        # going through each frame's result along with its timestamp
        for result, timestamp_sec in zip(results, timestamp_batch):

            # getting the names of all objects detected in this frame
            names_found = [model.names[int(c)] for c in result.boxes.cls]

            # keeping only the 5 objects required in our assignment
            names_found = [name for name in names_found if name in target_classes]

            # finding which second this frame belongs to
            bucket_idx = int(timestamp_sec)

            # counting how many people YOLO detected in this frame
            person_count_this_frame = sum(
                1 for c in result.boxes.cls
                if model.names[int(c)] == "person"
            )

            # removing duplicate object names because we only need to know
            # whether an object appeared in this frame or not
            objects_this_frame = set(names_found)

            # storing this frame's objects and people count in its bucket
            buckets[bucket_idx].append(
                (objects_this_frame, person_count_this_frame)
            )

    try:
        video_bytes = df.iloc[video_idx]["mp4"]

        with open("sample.mp4", "wb") as f:
            f.write(video_bytes)

        cap = cv2.VideoCapture("sample.mp4")

        if not cap.isOpened():
            raise ValueError("Could not open video")

        frame_num = 0
        frame_batch, timestamp_batch = [], []

        while frame_num < 60:
            ret, frame = cap.read()

            if not ret:
                break

            # getting the timestamp of the current frame in seconds
            timestamp_sec = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0

            # adding the frame and its timestamp to the current batch
            frame_batch.append(frame)
            timestamp_batch.append(timestamp_sec)

            frame_num += 1

            # once we have 16 frames, passing the whole batch to YOLO
            if len(frame_batch) == batch_size:
                process_batch(frame_batch, timestamp_batch)

                # clearing the batch so we can collect the next 16 frames
                frame_batch, timestamp_batch = [], []

        # processing the remaining frames if the last batch has less than 16
        if frame_batch:
            process_batch(frame_batch, timestamp_batch)

    except Exception as e:
        error_message = str(e)
        print(f"Video {video_idx} failed: {error_message}")

    finally:

        # closing the video after processing it
        if cap is not None:
            cap.release()

        # now combining all frame info in each bucket into one result per second
        if buckets:
            final_results = []

            for bucket_idx, records in sorted(buckets.items()):

                # getting the final objects and median people count for this second
                objects_present, median_people = aggregate_bucket(records)

                # creating one final entry for this second
                final_results.append({
                    "video": f"{video_filename}:{video_idx}",
                    "second": bucket_idx,
                    "objects": objects_present,
                    "people": median_people
                })

            # writing each second's result into results_batched.jsonl
            with open("results_batched.jsonl", "a") as out_file:
                for record in final_results:
                    out_file.write(json.dumps(record) + "\n")

        # writing an error record if something goes wrong
        if 'error_message' in locals():
            with open("results_batched.jsonl", "a") as out_file:
                error_record = {
                    "video": f"{video_filename}:{video_idx}",
                    "error": error_message
                }
                out_file.write(json.dumps(error_record) + "\n")

            del error_message

        else:
            print(f"Video {video_idx} processed successfully!")

Video 0 processed successfully!
Video 1 processed successfully!
Video 2 processed successfully!


In [22]:
# loading the results from a JSONL file and storing them in a dictionary
def load_results(path):
    out = {}

    with open(path) as f:
        for line in f:
            r = json.loads(line)

            # skipping error records because they don't contain detection results
            if "error" not in r:

                # using video name and second as the key to identify each result
                out[(r["video"], r["second"])] = (
                    sorted(r["objects"]),
                    r["people"]
                )

    return out


# loading results from both individual and batched processing
original = load_results("results.jsonl")
batched = load_results("results_batched.jsonl")

# checking which seconds have different results in the two versions
mismatches = [
    k for k in original
    if original[k] != batched.get(k)
]

# displaying how many results don't match
print(f"{len(mismatches)} mismatches out of {len(original)}")

0 mismatches out of 6


now we are testing different batch sizes to see which one gives us the best performance. we'll process the same 3 videos with batch sizes 1, 16, 32 and 64 and measure how much time each takes. this will help us compare individual frame processing with batching and see if increasing the batch size actually makes things faster

In [23]:
import time

def time_batch_size(batch_size, video_indices=[0, 1, 2]):
    start = time.time()

    for video_idx in video_indices:
        video_bytes = df.iloc[video_idx]["mp4"]
        with open("sample.mp4", "wb") as f:
            f.write(video_bytes)

        cap = cv2.VideoCapture("sample.mp4")
        frame_num = 0
        frame_batch = []

        while frame_num < 60:
            ret, frame = cap.read()
            if not ret:
                break
            frame_batch.append(frame)
            frame_num += 1

            if len(frame_batch) == batch_size:
                model(frame_batch, conf=0.4, verbose=False)
                frame_batch = []

        if frame_batch:
            model(frame_batch, conf=0.4, verbose=False)

        cap.release()

    elapsed = time.time() - start
    print(f"batch_size={batch_size}: {elapsed:.2f}s for {len(video_indices)} videos")
    return elapsed


for bs in [1, 16, 32, 64]:
    time_batch_size(bs)

batch_size=1: 5.54s for 3 videos
batch_size=16: 4.68s for 3 videos
batch_size=32: 4.91s for 3 videos
batch_size=64: 5.02s for 3 videos


since going from 16 to 32 isnt doing drastic changes in time. we will keep the batchsize 16

In [ ]:
# Final Pipeline
import cv2
import json
import os
import subprocess
import time
import pandas as pd
from collections import defaultdict

batch_size = 16
output_path = "final_results.jsonl"   # saving our final results in a separate file

all_file_names = [f"train-{i:05d}-of-01357.parquet" for i in range(50)]
file_names = all_file_names[4:6] # starting with just one file to test everything first


def process_video(video_bytes, video_filename, video_idx, out_file):
    buckets = defaultdict(list)
    cap = None

    def process_batch(frame_batch, timestamp_batch):

        # passing all frames in this batch to YOLO in one call
        results = model(frame_batch, conf=0.4, verbose=False)

        # going through each frame's result along with its timestamp
        for result, ts in zip(results, timestamp_batch):

            # getting the names of all objects detected in this frame
            names_found = [model.names[int(c)] for c in result.boxes.cls]

            # keeping only the 5 objects required in our assignment
            names_found = [n for n in names_found if n in target_classes]

            # finding which second this frame belongs to
            bucket_idx = int(ts)

            # counting how many people YOLO detected in this frame
            person_count = sum(
                1 for c in result.boxes.cls
                if model.names[int(c)] == "person"
            )

            # removing duplicate object names because we only need to know
            # whether an object appeared in this frame or not
            objects_this_frame = set(names_found)

            # storing this frame's objects and people count in its bucket
            buckets[bucket_idx].append((objects_this_frame, person_count))

    try:
        # writing the video bytes into a temporary mp4 file
        with open("sample.mp4", "wb") as f:
            f.write(video_bytes)

        cap = cv2.VideoCapture("sample.mp4")

        if not cap.isOpened():
            raise ValueError("Could not open video")

        frame_batch, timestamp_batch = [], []

        while True:
            ret, frame = cap.read()

            if not ret:
                break

            # getting the timestamp of the current frame in seconds
            timestamp_sec = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0

            # adding the frame and its timestamp to the current batch
            frame_batch.append(frame)
            timestamp_batch.append(timestamp_sec)

            # once we have 16 frames, passing the whole batch to YOLO
            if len(frame_batch) == batch_size:
                process_batch(frame_batch, timestamp_batch)

                # clearing the batch so we can collect the next 16 frames
                frame_batch, timestamp_batch = [], []

        # processing the remaining frames if the last batch has less than 16
        if frame_batch:
            process_batch(frame_batch, timestamp_batch)

    except Exception as e:
        error_message = str(e)
        print(f"Video {video_idx} failed: {error_message}")

    finally:
        # closing the video after processing it
        if cap is not None:
            cap.release()

        # now combining all frame info in each bucket into one result per second
        for bucket_idx, records in sorted(buckets.items()):

            # getting the final objects and median people count for this second
            objects_present, median_people = aggregate_bucket(records)

            # creating one final entry for this second
            record = {
                "video": f"{video_filename}:{video_idx}",
                "second": bucket_idx,
                "objects": objects_present,
                "people": median_people
            }

            # writing this second's result as a separate JSON line
            out_file.write(json.dumps(record) + "\n")

        # writing an error record if something goes wrong
        if 'error_message' in locals():
            error_record = {
                "video": f"{video_filename}:{video_idx}",
                "error": error_message
            }
            out_file.write(json.dumps(error_record) + "\n")


# downloading and processing the selected parquet files
with open(output_path, "a") as out_file:

    for file_name in file_names:
        local_path = f"finevideo/data/{file_name}"

        print(f"downloading {file_name}...")

        subprocess.run([
            "hf", "download", "HuggingFaceFV/finevideo",
            "--repo-type", "dataset",
            "--include", f"data/{file_name}",
            "--local-dir", "finevideo"
        ], check=True)

        # reading the downloaded parquet file
        df_file = pd.read_parquet(local_path)

        print(f"processing {file_name}: {len(df_file)} videos")

        # measuring how long it takes to process all videos in this file
        file_start = time.time()

        for video_idx in range(len(df_file)):
            process_video(
                df_file.iloc[video_idx]["mp4"],
                file_name,
                video_idx,
                out_file
            )
            print(f"  video {video_idx} done")

        file_elapsed = time.time() - file_start

        # deleting the parquet file to free up disk space
        os.remove(local_path)

        print(
            f"finished {file_name} in {file_elapsed / 60:.1f} min, "
            f"deleted to free space"
        )

print("done")

In [3]:
!pip freeze | grep -iE "ultralytics|opencv|pandas|pyarrow|torch"

geopandas==1.2.0
opencv-contrib-python==4.14.0.94
opencv-python==4.14.0.94
opencv-python-headless==4.14.0.94
pandas==2.3.3
pandas-datareader==0.11.1
pandas-gbq==0.30.0
pandas-profiling==3.6.6
pandas-stubs==2.2.3.250527
pandasql==0.7.3
pyarrow==23.0.1
pytorch-ignite==0.5.5
pytorch-lightning==2.6.6
sklearn-pandas==2.2.0
torch==2.11.0+cu130
torchao==0.10.0
torchaudio==2.11.0+cu130
torchcodec==0.11.0+cu130
torchdata==0.11.0
torchinfo==1.8.0
torchmetrics==1.9.0
torchsummary==1.5.1
torchtune==0.6.1
torchvision==0.26.0+cu130
ultralytics==8.4.216
ultralytics-platform==0.1.120
ultralytics-thop==2.2.5


Finding out what tooks the most time below:
results were: 
Inference is the bottleneck: it took 27.03 s (98.7%) of the time and decoding took only 0.34 s (1.3%) on 929 frames,

In [10]:

import pandas as pd
df = pd.read_parquet("finevideo/data/train-00001-of-01357.parquet")
with open("sample.mp4", "wb") as f:
    f.write(df.iloc[0]["mp4"])

In [11]:
import cv2
import time
from ultralytics import YOLO

video_path = "sample.mp4"   # change this to any local mp4 path you have
batch_size = 16
conf_threshold = 0.4

model = YOLO("yolo11l.pt")

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise ValueError(f"Could not open {video_path}")

decode_time = 0.0
infer_time = 0.0
frame_batch = []
total_frames = 0

while True:
    t0 = time.time()
    ret, frame = cap.read()
    decode_time += time.time() - t0

    if not ret:
        break

    total_frames += 1
    frame_batch.append(frame)

    if len(frame_batch) == batch_size:
        t0 = time.time()
        model(frame_batch, conf=conf_threshold, verbose=False)
        infer_time += time.time() - t0
        frame_batch = []

if frame_batch:
    t0 = time.time()
    model(frame_batch, conf=conf_threshold, verbose=False)
    infer_time += time.time() - t0

cap.release()

total = decode_time + infer_time
print(f"total frames: {total_frames}")
print(f"decode time:    {decode_time:.2f}s ({100*decode_time/total:.1f}%)")
print(f"inference time: {infer_time:.2f}s ({100*infer_time/total:.1f}%)")

total frames: 929
decode time:    0.34s (1.3%)
inference time: 27.03s (98.7%)
